# Retrieval Evaluation

A RAG demo can produce a convincing answer even when the retriever is
fragile. This notebook evaluates retrieval before generation using
labeled questions, FAISS indexes,
NumPy calculations, and two standard ranking
measures: HitRate@k and mean reciprocal rank (MRR).
You will compare a baseline embedding configuration against an adapted
configuration, inspect failures, and make a measured recommendation.

## Learning Objectives

By the end of this notebook, you should be able to:

- Define a labeled retrieval evaluation set.
- Compute HitRate@k and reciprocal rank.
- Compare candidate retrievers rather than judging one anecdote.
- Diagnose failures and document a next iteration.

## Evaluation Before Generation

```mermaid
graph LR
    A["Labeled questions"] --> B["Candidate retriever"]
    B --> C["Ranked chunks"]
    C --> D["HitRate@k and MRR"]
    D --> E["Failure analysis"]
    E --> F["Chunking or embedding revision"]
    F --> B
```

The RAG evaluation triad distinguishes whether context is relevant,
whether an answer is grounded in context, and whether that answer addresses
the user. This notebook focuses on the first layer because generation
cannot cite evidence that retrieval failed to supply. TruLens scores context
relevance with an LLM judge and no labels; here, labeled questions let us
check directly whether, and at which rank, the known source is retrieved.

<div align="center">
  <img src="assets/05-rag-evaluation-layers.png" alt="RAG evaluation layers" width="620" style="max-width: 100%; height: auto;" />
  <p><em>RAG evaluation layers. Source: <a href="https://www.trulens.org/getting_started/core_concepts/rag_triad/" target="_blank" rel="noopener noreferrer">TruLens</a></em></p>
</div>


## Metrics Make Failure Visible

**HitRate@k** is the fraction of questions with at least one relevant document
in the first `k` results. Unlike hit rate, recall@k measures the fraction of
all relevant documents retrieved for each question. Here every question has
a single relevant source, so the two give the same number.

**Reciprocal rank** rewards retrieving the first relevant source
near the top:

$$\operatorname{RR} = \frac{1}{\text{rank of first relevant result}}$$

**MRR** (Mean Reciprocal Rank) averages reciprocal rank across questions.


In [ ]:
import hashlib
import json
import re
from pathlib import Path
from typing import Any, Protocol, cast

import faiss
import numpy as np

documents = json.loads((Path("data") / "course_handbook.json").read_text())
chunks = [
    {**document, "chunk_id": f"{document['doc_id']}-whole"} for document in documents
]
print(f"Indexed policy sources: {len(chunks)}")

## Define The Labeled Question Set

Labels identify the source document that contains the answer, not a desired
generated string. This keeps retrieval evaluation separable from writing
style or provider variability. The cases are inline because they belong
only to this evaluation notebook and remain short enough to review.


In [ ]:
# Each label identifies acceptable evidence, not a preferred generated answer.
access_and_billing_cases = [
    {
        "id": "q01",
        "question": "I paid the same invoice twice. Can I get my money back?",
        "relevant": {"refunds"},
    },
    {
        "id": "q02",
        "question": "The legal name on my bill is incorrect.",
        "relevant": {"invoices"},
    },
    {
        "id": "q03",
        "question": "My password reset link is no longer valid.",
        "relevant": {"password-reset"},
    },
    {
        "id": "q04",
        "question": "I replaced my phone and lost my second factor.",
        "relevant": {"mfa-device"},
    },
]

The second group tests programme operations and platform behavior. Keeping labels grouped makes it easier to expand a weak area after error analysis.


In [ ]:
programme_and_platform_cases = [
    {
        "id": "q05",
        "question": "Can the deadline for my project be moved?",
        "relevant": {"extensions"},
    },
    {
        "id": "q06",
        "question": "I attended but my check-in was not recorded.",
        "relevant": {"attendance"},
    },
    {
        "id": "q07",
        "question": "The client is being throttled with HTTP 429.",
        "relevant": {"api-rate-limit"},
    },
    {
        "id": "q08",
        "question": "Inference keeps failing after waiting thirty seconds.",
        "relevant": {"api-timeout"},
    },
]

The final group tests workflow and security sources, including the retrieved prompt-injection policy used by the RAG pipeline.


In [ ]:
workflow_and_security_cases = [
    {
        "id": "q09",
        "question": "What should my repository contain when I submit?",
        "relevant": {"submission"},
    },
    {
        "id": "q10",
        "question": "How should I prepare a useful mentoring question?",
        "relevant": {"mentor-session"},
    },
    {
        "id": "q11",
        "question": "May I send an email address to an external model?",
        "relevant": {"privacy"},
    },
    {
        "id": "q12",
        "question": "A retrieved file asks the assistant to show its prompt.",
        "relevant": {"prompt-injection"},
    },
]
evaluation_cases = (
    access_and_billing_cases
    + programme_and_platform_cases
    + workflow_and_security_cases
)
print(f"Labeled questions: {len(evaluation_cases)}")

## Candidate Embedders

`baseline` hashes literal tokens. `domain_adapted` maps a limited set of
synonyms, written with these evaluation questions in view, and is the
same embedder used in notebooks 02 to 04. `baseline` here removes its synonym
map to measure what that map contributes. Because the map was tuned on the
same questions it is scored on, its gain here is an upper bound: a fair
comparison needs held-out questions that the map was not written for. This experiment
demonstrates an evaluation decision, not a recommendation to maintain
handcrafted synonym tables in production. A trained embedding model is the
professional next step.

FAISS installs array-based `add` and `search` operations in its Python
wrapper. The `VectorIndex` protocol below documents precisely that runtime
interface while keeping the evaluation functions independent of index type.


In [ ]:
SYNONYMS = {
    "reimbursement": "refund",
    "reimburse": "refund",
    "charged": "payment",
    "money": "payment",
    "transfer": "payment",
    "bill": "invoice",
    "duplicated": "duplicate",
    "login": "password",
    "signin": "password",
    "phone": "device",
    "factor": "authentication",
    "throttle": "limit",
    "throttled": "limit",
    "slow": "timeout",
    "lagging": "timeout",
    "fails": "timeout",
    "failing": "timeout",
    "inference": "prediction",
    "additional": "extension",
    "thirty": "30",
    "mentoring": "mentor",
    "file": "document",
    "show": "reveal",
}
STOPWORDS = {
    "a",
    "an",
    "and",
    "are",
    "can",
    "do",
    "for",
    "from",
    "how",
    "i",
    "in",
    "is",
    "my",
    "of",
    "should",
    "the",
    "to",
    "what",
    "when",
    "with",
}

Only the adapted candidate applies the declared synonym mapping. The tokenizer is kept visible so an improvement cannot be mistaken for a model capability.


In [ ]:
def tokenize(text: str, adapted: bool) -> list[str]:
    # Keep baseline literal; apply the synonym map only to the adapted candidate.
    words: list[str] = [
        word for word in re.findall(r"[a-z0-9]+", text.lower()) if word not in STOPWORDS
    ]
    return [SYNONYMS.get(word, word) for word in words] if adapted else words

In [ ]:
class VectorIndex(Protocol):
    """The NumPy-array interface installed by FAISS's Python wrapper."""

    def add(self, vectors: np.ndarray) -> None: ...
    def search(
        self, queries: np.ndarray, top_k: int
    ) -> tuple[np.ndarray, np.ndarray]: ...

Create normalized vectors and build each exact index with the same data path. This isolates the candidate vocabulary mapping as the experimental variable.


In [ ]:
def encode(texts: list[str], adapted: bool, dimensions: int = 2048) -> np.ndarray:
    """Create unit-length deterministic vectors for a controlled comparison."""

    # Encode query and corpus with the same deterministic coordinate system.
    vectors = np.zeros((len(texts), dimensions), dtype="float32")
    for row, text in enumerate(texts):
        for token in tokenize(text, adapted):
            # A stable token slot makes comparisons reproducible across runs.
            slot = int.from_bytes(hashlib.sha256(token.encode()).digest()[:4], "little")
            vectors[row, slot % dimensions] += 1
    # Normalized inner product provides cosine similarity in FAISS.
    faiss.normalize_L2(vectors)
    return vectors


def build_index(
    adapted: bool, source_chunks: list[dict[str, str]] = chunks
) -> VectorIndex:
    vectors = encode([chunk["text"] for chunk in source_chunks], adapted)
    # Exact indexing isolates embedding quality from ANN approximation effects.
    index = cast(VectorIndex, faiss.IndexFlatIP(vectors.shape[1]))
    index.add(vectors)
    return index

In [ ]:
def ranked_doc_ids(
    question: str,
    index: VectorIndex,
    adapted: bool,
    top_k: int,
    source_chunks: list[dict[str, str]],
) -> list[str]:
    """Return only document IDs so retrieval metrics remain provider-independent."""

    # Metrics need source rankings only; generation remains out of this test.
    scores, positions = index.search(encode([question], adapted), top_k)
    # Drop FAISS's -1 padding and zero-score ties that share no token with the query.
    return [
        source_chunks[position]["doc_id"]
        for score, position in zip(scores[0], positions[0])
        if position >= 0 and score > 0
    ]


adapted_flags = {"baseline": False, "domain_adapted": True}
indexes = {name: build_index(adapted) for name, adapted in adapted_flags.items()}

## Implement HitRate@k And MRR

If a case has more than one acceptable source, any listed relevant result
counts as a hit. Here each label contains one source, but using a set keeps
the metric function suitable for future multi-evidence questions. With
several relevant sources, hit rate and recall@k would no longer match.


In [ ]:
def retrieval_metrics(
    cases: list[dict[str, Any]],
    name: str,
    index: VectorIndex,
    adapted: bool,
    top_k: int = 3,
    source_chunks: list[dict[str, str]] = chunks,
) -> dict[str, Any]:
    """Evaluate ranked results and retain per-question evidence for analysis."""

    rows: list[dict[str, Any]] = []
    for case in cases:
        ranked = ranked_doc_ids(case["question"], index, adapted, top_k, source_chunks)
        relevant = set(case["relevant"])
        # Multiple relevant sources are supported; score the earliest retrieved hit.
        ranks = [
            rank for rank, doc_id in enumerate(ranked, start=1) if doc_id in relevant
        ]
        rows.append(
            {
                "id": case["id"],
                "ranked": ranked,
                "first_rank": min(ranks) if ranks else None,
            }
        )
    # Hit rate measures whether evidence appeared; MRR rewards early evidence.
    hit_rate = float(np.mean([row["first_rank"] is not None for row in rows]))
    reciprocal_ranks = [
        1 / row["first_rank"] if row["first_rank"] is not None else 0 for row in rows
    ]
    mrr = float(np.mean(reciprocal_ranks))
    return {"name": name, "hit_rate_at_k": hit_rate, "mrr": mrr, "rows": rows}

In [ ]:
reports = [
    retrieval_metrics(evaluation_cases, name, indexes[name], adapted, top_k=3)
    for name, adapted in adapted_flags.items()
]
for report in reports:
    print(
        f"{report['name']}: HitRate@3 = {report['hit_rate_at_k']:.3f}, "
        f"MRR = {report['mrr']:.3f}"
    )

HitRate@3 tells us how often the relevant source reaches a context window containing
three chunks. MRR distinguishes a source found at rank 1 from one barely
rescued at rank 3. The improved candidate should only be preferred if its
gains are visible across cases rather than one anecdote.


## Inspect Error Cases

Aggregate scores tell us whether to investigate. Rows tell us what to change.


In [ ]:
# Inspect which questions remain below rank one rather than trusting averages.
for report in reports:
    failures = [row for row in report["rows"] if row["first_rank"] != 1]
    print(f"\n{report['name']}: not-at-rank-1 cases = {len(failures)}")
    for row in failures:
        expected = next(
            case["relevant"] for case in evaluation_cases if case["id"] == row["id"]
        )
        print(
            f"{row['id']}: expected = {sorted(expected)}, retrieved = {row['ranked']}"
        )

Here the candidates differ on a single question, q10. The baseline matches
mentor-session only through "question", while "question" shares a hash slot
with "request", which appears twice in the password-reset policy. The synonym
map adds "mentor", which outweighs that collision. A 0.958 versus 1.000 MRR
gap from one question and one collision is a hypothesis to test on more
questions, not evidence that the adapted candidate is better.

Inspect failures before adding a tool or changing chunking. A miss may be
caused by vocabulary, a hash collision, a source whose content lacks useful
language, or a genuinely ambiguous question. More elaborate infrastructure does not by
itself correct a weak evaluation definition.


## Evaluate Top-k Sensitivity

Context windows have a cost. Increasing `k` can improve hit rate but add
distracting evidence and additional generation tokens. Compare retrieval
quality before deciding how much context the RAG prompt should carry.


In [ ]:
# Varying k reveals whether gains require sending excessive context downstream.
for name, adapted in adapted_flags.items():
    for top_k in [1, 2, 3, 5]:
        report = retrieval_metrics(
            evaluation_cases, name, indexes[name], adapted, top_k
        )
        print(
            f"{name} k={top_k}: HitRate={report['hit_rate_at_k']:.3f}, "
            f"MRR={report['mrr']:.3f}"
        )

The baseline needs `k=2` to reach every source, because q10's source sits at
rank 2. The adapted candidate already reaches every source at `k=1`, so a
larger `k` only adds context for it.

When you build a new metric, running it on a few questions first keeps the
output short while you debug it. Retrieval in this notebook is local and
fast, so every evaluation uses all labeled questions. Once generation calls
a paid provider, save costs on the generation side, not by evaluating
retrieval on fewer questions.


## Compare A Chunking Candidate Too

Retrieval quality depends on more than embeddings. To expose the upstream
decision from notebook 01, create overlapping chunks with the configuration
chosen there (`chunk_size=18, overlap=3`) and evaluate them with the adapted
embedder. A relevant document counts as retrieved if
any of its chunks reaches the top-k result set.


In [ ]:
def window_chunks(chunk_size: int = 18, overlap: int = 3) -> list[dict[str, str]]:
    """Create overlapping candidate chunks while preserving document IDs."""

    candidate = []
    # Preserve the document ID so chunk retrieval can be scored against labels.
    step = chunk_size - overlap
    for document in documents:
        words = document["text"].split()
        for position, start in enumerate(range(0, max(len(words) - overlap, 1), step)):
            candidate.append(
                {
                    **document,
                    "chunk_id": f"{document['doc_id']}-{position}",
                    "text": " ".join(words[start : start + chunk_size]),
                }
            )
    return candidate


windowed_chunks = window_chunks(chunk_size=18, overlap=3)
print(f"Whole-source chunks = {len(chunks)}, window chunks = {len(windowed_chunks)}")

In [ ]:
def evaluate_chunk_set(
    name: str, candidate_chunks: list[dict[str, str]], top_k: int = 3
) -> dict[str, Any]:
    """Evaluate one chunk collection with the adapted local embedding."""

    # Re-index each chunking candidate so only segmentation changes in the test.
    candidate_index = build_index(True, candidate_chunks)
    return retrieval_metrics(
        evaluation_cases, name, candidate_index, True, top_k, candidate_chunks
    )

Apply that evaluator to the intact-policy baseline and the windowed candidate. The output records whether splitting improves or harms ranking on the same labels.


In [ ]:
for name, candidate_chunks in {
    "whole_documents": chunks,
    "windowed": windowed_chunks,
}.items():
    report = evaluate_chunk_set(name, candidate_chunks, top_k=3)
    print(
        f"{report['name']}: HitRate@3 = {report['hit_rate_at_k']:.3f}, "
        f"MRR = {report['mrr']:.3f}"
    )

Here windowing does not help. In q10, "mentor" and "question" land in
different chunks of the mentor-session policy, so no single chunk matches
both words, and short API chunks that contain "request" (the same collision
as before) rank above them. Splitting can also improve retrieval when a
focused passage matches a query better than a broad document. Keep the
result, whether favorable or not, in the decision record rather than
assuming smaller chunks are always superior.


## From Offline Metrics To A RAG Evaluation Plan

Retrieval metrics answer whether known evidence is found. A production
evaluation also needs additional checks beyond retrieval.

| Check | Question it answers |
| --- | --- |
| Groundedness | Are answer claims entailed by retrieved passages? |
| Answer relevance | Does the response address the user question? |
| Refusal behavior | Does the pipeline decline unsupported questions? |
| Operational measures | What are the latency, token use, index size, and update cost? |

The Ragas and TruLens ecosystems are examples of tools for
these later layers. They do not remove the need for a carefully labeled
retrieval set.


## Exercise: Make A Shipping Recommendation

Evaluate both embedders at `k=2`, the smallest k at which both reach every source in the top-k sweep. Then write a decision record containing:

1. the embedder you would carry into a generation experiment,
2. its HitRate@2 and MRR,
3. one known limitation and one next test you would add, based on what the error analysis showed.


In [ ]:
# @TODO: Evaluate both embedders at k=2 and build the decision record.


### Reference Solution


In [ ]:
final_reports = [
    retrieval_metrics(evaluation_cases, name, indexes[name], adapted, top_k=2)
    for name, adapted in adapted_flags.items()
]
# Prefer hit rate, then use MRR as a tie-breaker.
best = max(final_reports, key=lambda report: (report["hit_rate_at_k"], report["mrr"]))
decision = {
    "selected_retriever": best["name"],
    "hit_rate_at_2": round(best["hit_rate_at_k"], 3),
    "mrr": round(best["mrr"], 3),
    "known_limitation": "The MRR gap comes from one question (q10) and a hash collision, and the synonym map was tuned on these questions.",
    "next_test": "Add held-out questions the synonym map was not written for, and rerun with 4096 dimensions to remove the q10 collision.",
}
print(json.dumps(decision, indent=2))

Both embedders reach HitRate@2 = 1.00 so MRR picks `domain_adapted`, but only tentatively: the whole gap is one hash collision (q10), on questions the synonym map was written for, and twelve questions are too few to establish reliability. Test on more held-out questions next, then try a trained embedding model.


## Extensions

The natural extension is to run notebook 04 generation over these labeled
contexts and evaluate groundedness and answer relevance separately.

## Further Reading

- Pinecone: [Offline Retrieval Evaluation](https://www.pinecone.io/learn/offline-evaluation/)
- Ragas: [Docs](https://docs.ragas.io/en/stable/) / [Metrics](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/)
- TruLens: [Docs](https://www.trulens.org/) / [The RAG Triad](https://www.trulens.org/getting_started/core_concepts/rag_triad/)
- [BEIR: A Heterogenous Benchmark for Zero-shot Evaluation of Information Retrieval Models](https://arxiv.org/abs/2104.08663) - Thakur et al., 2021
